ERA5 reanalysis atmospheres
===========================

MCEq can run on a **measured or reanalysed atmosphere** instead of a
parametrization, through
`MCEq.geometry.density_profiles.TabulatedAtmosphere` and its location-centred
sibling `TabulatedLocationCentered`. They read a CSV table; converting a
dataset into that table is left to the user, because every archive has its own
API and file format and MCEq does not want to depend on any of them.

This notebook is the worked example for **ERA5**, the ECMWF reanalysis. It

1. turns the ERA5 **surface geopotential** into a table of ground elevations —
   the field that says where each atmospheric column actually stops,
2. converts an ERA5 netCDF download into MCEq tables — one per day, each a
   global longitude/latitude grid of vertical columns,
3. runs MCEq on them and compares against NRLMSISE-00,
4. maps and animates the result globally, with every plotted number coming
   out of the MCEq atmosphere interface rather than straight from the file,
5. shows why the grid matters for neutrinos: at large zenith angles, and for
   upgoing events in particular, the shower develops in a completely different
   column than the one above the detector.

> **Not executed when the documentation is built.** It needs a CDS account and
> a multi-hundred-MB download. Run it locally.

What you need
-------------

```bash
pip install cdsapi xarray netcdf4 dask cartopy
```

(`dask` is what lets `open_dataset(..., chunks=...)` below read a
multi-hundred-MB file lazily; cartopy downloads Natural Earth coastline data
the first time it draws a map, so the first plot needs a network connection.)

and, for the download, a CDS account with an API key in `~/.cdsapirc`
(register at <https://cds.climate.copernicus.eu>, then follow
<https://cds.climate.copernicus.eu/how-to-api> and accept the dataset licence
once from its download page).

The table format
----------------

A table is a CSV file. In its simplest form it is **one vertical column**:

```
# MCEq tabulated atmosphere v1
h_cm,T_K,p_hPa
283400.0,247.1,681.2
510000.0,231.4,500.0
900000.0,,300.0
```

* `h_cm` is required: height above sea level in cm.
* Density comes either from a `rho_gcm3` column directly, or from `T_K` and
  `p_hPa` via the dry-air ideal gas law.
* `T_K` also feeds `get_temperature()`, `p_hPa` feeds `get_pressure()`.
* Comment lines start with `#`, column order does not matter, unknown columns
  are ignored, rows may be in any order, and missing values are an empty field
  or `nan`.

Adding **`lat_deg` and `lon_deg`** turns it into a *grid* of columns — one row
per (grid node, level), long format:

```
lat_deg,lon_deg,h_cm,T_K,p_hPa
-90.0,0.0,110.9,242.8,1000
-90.0,0.0,1352.0,241.6,975
...
```

The nodes must form a regular longitude/latitude grid, and every column must
carry the same number of levels. That is exactly how ERA5 pressure-level data
is shaped, so the conversion is a reshape.

In [ ]:
import gzip
import os

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

G0 = 9.80665  # standard gravity, m/s^2

# --- input ----------------------------------------------------------------
# An ERA5 pressure-level download: "ERA5 daily statistics on pressure levels"
# (daily mean), variables *temperature* and *geopotential*, all 37 levels,
# global. The CDS portal delivers one netCDF file per variable, so point this
# at both; a single file holding both variables works just as well.
NC_FILES = [
    "geopotential_stream-oper_daily-mean.nc",
    "temperature_0_daily-mean.nc",
]

# The ground. ERA5's surface geopotential is one of its *invariant* fields --
# it is the model's fixed orography, so a single timestep covers the whole
# archive and this file never has to be downloaded again:
#
#   c.retrieve("reanalysis-era5-single-levels", {
#       "product_type": "reanalysis", "variable": ["geopotential"],
#       "year": "2013", "month": "08", "day": "09", "time": "12:00",
#       "data_format": "netcdf"}, OROGRAPHY_FILE)
#
# That returns the 0.25 deg field. ECMWF also publishes the orography of the
# operational Tco1279 grid ("geo_1279l4_0.1x0.1.grib2") at 0.1 deg, finer than
# ERA5's own T639 (~31 km) terrain. Either file works here: the elevations are
# *aggregated* over each atmosphere cell below rather than point-sampled, so
# the answer does not depend on the grid they arrive on.
#
# Beware the name collision: the *pressure-level* z in NC_FILES is the height
# of each pressure surface and moves with the weather; the *single-level* z
# here is the terrain and does not.
OROGRAPHY_FILE = "era5_orography.nc"

# How much of the native 0.25 deg grid of *atmospheric* columns to keep. The
# ground is a separate field and is always aggregated from its own native
# resolution, which is where resolution actually matters -- see "The ground"
# below. One column costs about 0.35 ms; a CSV row costs ~44 bytes and there is
# one per level per column.
#
#   STRIDE  grid       columns    map      table (gzipped)
#       16  4 deg        4 140     1.4 s      1.7 MB / day
#        4  1 deg       65 160      24 s       26 MB / day
#        2  0.5 deg    259 920      93 s     ~100 MB / day
#        1  0.25 deg     1.04 M    ~6 min    ~410 MB / day
#
# The last two table sizes and the 0.25 deg timing are extrapolated; the cost
# per column is flat at 0.34-0.36 ms across every grid measured above.
#
# 1 deg is the right choice for a global run, for two reasons that have nothing
# to do with how the figure looks:
#
# * MCEq is one-dimensional. It develops the whole shower in the single column
#   at the impact point, and at 80 deg zenith the shower maximum sits ~126 km
#   downrange of that point (446 km at 89 deg). Sampling the atmosphere every
#   0.25 deg -- 22 km -- is finer than the approximation being fed.
# * Measured: taking column depth from the 1 deg grid and interpolating back,
#   instead of reading the native grid, is worth 0.3 g/cm^2 (p95) over the
#   Mediterranean and 1.7 g/cm^2 over Tibet, against a day-to-day spread of
#   ~8 g/cm^2. Point-sampling the *ground* onto that same grid costs 27 g/cm^2
#   at the 95th percentile over land and 167 g/cm^2 at worst -- an order of
#   magnitude more, and it grows with the stride. That is why the ground is
#   aggregated from its native grid and only the atmosphere is strided.
#
# STRIDE = 1 is not reachable globally through this CSV path in any case:
# 38.4 M rows is 1.7 GB, and AtmosphereTable.load_from_csv peaks at ~795 bytes per row,
# i.e. ~30 GB of RAM. Crop to a region and keep every point instead.
STRIDE = 4

OUTPUT_DIR = "era5_tables"

# --- the detector ---------------------------------------------------------
# KM3NeT/ARCA: off Capo Passero, and far enough from the pole that the azimuth
# angle genuinely changes which column the shower develops in.
SITE_NAME = "KM3NeT-ARCA"
SITE_LON, SITE_LAT = 16.1, 36.267
SITE_DEPTH_M = 3500.0
SITE_ELEVATION_M = 0.0

In [ ]:
# merge() aligns the per-variable files on their shared coordinates and fails
# loudly if the requests behind them did not use the same grid or dates.
ds = xr.merge(
    [xr.open_dataset(path, chunks={"valid_time": 1}) for path in NC_FILES],
    join="exact",
    compat="override",  # the files hold different variables; nothing to reconcile
)
print(ds)

HAS_GEOPOTENTIAL = "z" in ds.data_vars
print(f"\ngeopotential present: {HAS_GEOPOTENTIAL}")

## The ground

Everything else in this notebook sits on top of this field, so it comes first.

ERA5's pressure levels do not stop at the terrain: below it they are
**extrapolated**, and those levels are fictitious. Over the Antarctic plateau
the 1000 hPa surface comes out at $-55$ m, some 2.9 km *below* the ice. Nothing
in a pressure-level file marks where the ground is, so without this field every
column would have to start at sea level and the overburden over any mountain
would be badly wrong.

The surface geopotential fixes that, and it is cheap: it is an **invariant**
field — the model's own orography — so one timestep covers the whole of ERA5.
Converted to a height, $h_\mathrm{surface} = z_\mathrm{surface}/g_0$, it is
written here to its own small table, indexed by longitude and latitude just
like the atmosphere tables and read back with the same idea.

**Its resolution is kept, not thrown away.** The elevation is read at the
orography's own resolution and then *aggregated* over each atmosphere cell,
whatever `STRIDE` is. Keeping the one node nearest each cell centre instead —
the obvious thing — is the largest resolution error in this notebook, by one
to two orders of magnitude. Inside a 1° cell, sampled from the 0.1° orography,
the terrain spans a median 234 m and up to 4.9 km, and the node nearest the
centre differs from the cell mean by 20 m for the median land cell and 257 m
at the 95th percentile. Read back through MCEq that is a column depth wrong by
27 g/cm² at the 95th percentile and by up to 167 g/cm² in the mountains —
against the 0.3–1.7 g/cm² that striding the *atmosphere* from 0.25° to 1°
costs. Both figures grow with the cell: at 4° the elevation miss is 61 m and
518 m.

The averaging is done in $X$ rather than in $h$: $X(h)$ is close to
exponential, so the depth at the mean elevation is not the mean depth, and over
rough terrain the two differ by a few more g/cm².

In [ ]:
oro = xr.open_dataset(OROGRAPHY_FILE)
if "time" in oro.dims:
    oro = oro.isel(time=0)  # invariant: any timestep will do

# The native orography, on an ascending latitude axis and longitudes in
# [0, 360). ERA5 counts latitude north-to-south while AtmosphereTable.load_from_csv
# sorts it south-to-north, and getting that backwards puts the Antarctic
# plateau in the Arctic Ocean.
native_lat = oro["latitude"].values.astype(float)
native_lon = oro["longitude"].values.astype(float) % 360.0
elevation_native_m = (oro["z"] / G0).values.astype(float)
if native_lat[0] > native_lat[-1]:
    native_lat, elevation_native_m = native_lat[::-1], elevation_native_m[::-1]
lon_order = np.argsort(native_lon)
native_lon, elevation_native_m = native_lon[lon_order], elevation_native_m[:, lon_order]

# The grid the atmosphere tables will use.
grid_lat = np.sort(ds["latitude"].values[::STRIDE].astype(float))
grid_lon = np.sort(ds["longitude"].values[::STRIDE].astype(float) % 360.0)


def cell_lookup(node_axis, native_axis, periodic=False):
    """Native indices falling inside the cell each node stands for.

    A node represents the cell centred on it, half a grid step wide either
    side. Returns one index array per node, so a cell holds however many
    native points it holds -- 100 for a 1 deg cell on a 0.1 deg orography,
    half that for the clipped cells at the poles, 1 if the two grids match.

    Args:
      node_axis: coarse axis, ascending
      native_axis: fine axis, ascending and uniform
      periodic: True for longitude, whose end cells wrap through 360 deg
    """
    width = float(np.mean(np.diff(node_axis)))
    if periodic:
        step = float(np.mean(np.diff(native_axis)))
        count = max(int(round(width / step)), 1)
        start = np.searchsorted(native_axis, (node_axis - width / 2) % 360.0)
        return [(s + np.arange(count)) % native_axis.size for s in start]
    lo = np.searchsorted(native_axis, node_axis - width / 2)
    hi = np.searchsorted(native_axis, node_axis + width / 2)
    return [np.arange(a, max(b, a + 1)) for a, b in zip(lo, hi)]


grid_jj = cell_lookup(grid_lat, native_lat)
grid_ii = cell_lookup(grid_lon, native_lon, periodic=True)

# Per-cell statistics. The mean is what the columns are built on; the spread is
# what keeping a single point would have discarded. h_point_m is that single
# point -- the native node nearest the cell centre -- kept so the cost of
# point sampling can be shown further down rather than asserted.
near_j = [int(np.argmin(np.abs(native_lat - value))) for value in grid_lat]
near_i = [int(np.argmin(np.abs(native_lon - value))) for value in grid_lon]
elev_point = elevation_native_m[np.ix_(near_j, near_i)]

shape = (grid_lat.size, grid_lon.size)
elev_mean, elev_min = np.empty(shape), np.empty(shape)
elev_max, elev_std = np.empty(shape), np.empty(shape)
for j, jj in enumerate(grid_jj):
    band = elevation_native_m[jj]
    for i, ii in enumerate(grid_ii):
        cell = band[:, ii]
        elev_mean[j, i] = cell.mean()
        elev_min[j, i] = cell.min()
        elev_max[j, i] = cell.max()
        elev_std[j, i] = cell.std()

SURFACE_TABLE = os.path.join(OUTPUT_DIR, "era5_surface.csv")
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(SURFACE_TABLE, "w") as out:
    out.write("# ERA5 surface elevation, from the invariant surface geopotential\n")
    out.write(f"# {native_lat.size}x{native_lon.size} native nodes aggregated "
              f"onto {grid_lat.size}x{grid_lon.size} cells\n")
    out.write("lat_deg,lon_deg,h_surface_m,h_min_m,h_max_m,h_std_m,h_point_m\n")
    np.savetxt(
        out,
        np.column_stack([
            np.repeat(grid_lat, grid_lon.size),
            np.tile(grid_lon, grid_lat.size),
            elev_mean.reshape(-1), elev_min.reshape(-1), elev_max.reshape(-1),
            elev_std.reshape(-1), elev_point.reshape(-1),
        ]),
        delimiter=",", fmt="%.4f,%.4f,%.2f,%.2f,%.2f,%.2f,%.2f",
    )

print(f"{SURFACE_TABLE}  {os.path.getsize(SURFACE_TABLE) / 1e6:.1f} MB")
print(f"native grid        {native_lat.size} x {native_lon.size} "
      f"({abs(np.diff(native_lat)[0]):.2f} deg)")
print(f"cell grid          {grid_lat.size} x {grid_lon.size} "
      f"({abs(np.diff(grid_lat)[0]):.2f} deg)")
print(f"native elevation   {elevation_native_m.min():8.1f} .. "
      f"{elevation_native_m.max():.1f} m")
print(f"cell means         {elev_mean.min():8.1f} .. {elev_mean.max():.1f} m")

# What the aggregation is worth, in metres of ground.
land = elev_mean > 1.0
span = (elev_max - elev_min)[land]
miss = np.abs(elev_point - elev_mean)[land]
K = 0.12  # g/cm^2 per metre: dX/dh = rho ~ 1.2e-3 g/cm^3 near the surface
print(f"\ninside one cell, over land ({land.sum()} cells):")
print(f"  terrain span   median {np.median(span):6.0f} m   "
      f"p95 {np.percentile(span, 95):6.0f} m   max {span.max():6.0f} m")
print(f"  point - mean   median {np.median(miss):6.0f} m   "
      f"p95 {np.percentile(miss, 95):6.0f} m")
print(f"                 ~{np.median(miss) * K:.0f} and "
      f"{np.percentile(miss, 95) * K:.0f} g/cm^2 of overburden at "
      f"dX/dh ~ {K} g/cm^2 per m")

### The surface elevation of the Earth

Plotted at the orography's native resolution, finer than the grid the
atmosphere tables use. Reanalysis orography is smoothed, so peaks come out
lower than the real summits — the South Pole reads about 2765 m against
IceCube's 2835 m, and the Himalaya tops out near 6000 m rather than 8850 m —
and it is smoothed differently in the 0.1° and 0.25° files. For a detector,
use the site's surveyed elevation rather than either. The cell means computed
above are for the global maps, where no single point is the right answer.

In [ ]:
import warnings

import cartopy.crs as ccrs
import matplotlib as mpl
from cartopy.util import add_cyclic_point

# Cartopy clips coastline polygons at the edge of the Robinson projection and
# shapely grumbles about the empty geometries that produces. Harmless, loud.
warnings.filterwarnings(
    "ignore", message="invalid value encountered in create_collection"
)


def paled(name, amount=0.55):
    """A washed-out version of a colormap, for use as a backdrop.

    Drawing the mesh semi-transparent instead would be the obvious way, but
    add_cyclic_point's wrapped column lands back on longitude 0 in a Robinson
    projection and any alpha < 1 makes that overlap show up as a seam down the
    middle of the map. Fading the colours and staying opaque avoids it.
    """
    colors = mpl.colormaps[name](np.linspace(0.0, 1.0, 256))
    colors[:, :3] = 1.0 - amount * (1.0 - colors[:, :3])
    return mpl.colors.ListedColormap(colors)


def global_map(ax, field, lat, lon, cmap, label, shrink=0.85, **kwargs):
    """Draws a lon/lat field on a Robinson projection."""
    # add_cyclic_point insists the longitude axis be equally spaced, which a
    # float32 axis is not, quite: ERA5's 0.2 deg steps come back as 0.199982 to
    # 0.200012. Rebuild it from its endpoints when it is uniform to that sort
    # of tolerance, and leave it alone -- to raise -- when it genuinely is not.
    lon = np.asarray(lon, dtype=float)
    steps = np.diff(lon)
    if np.ptp(steps) < 1e-3 * np.abs(steps.mean()) * len(lon):
        lon = np.linspace(lon[0], lon[-1], lon.size)
    cyclic, lon_c = add_cyclic_point(field, coord=lon)
    mesh = ax.pcolormesh(
        lon_c, lat, cyclic, transform=ccrs.PlateCarree(),
        cmap=cmap, shading="auto", **kwargs
    )
    ax.coastlines(lw=0.4, color="0.3")
    ax.set_global()
    plt.colorbar(mesh, ax=ax, orientation="horizontal", pad=0.04, label=label,
                 shrink=shrink, aspect=40, extend="both")
    return mesh


# Every second native point is plenty for a global figure.
fine = elevation_native_m[::2, ::2]

fig = plt.figure(figsize=(9, 5.0), dpi=120)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())
global_map(ax, fine, native_lat[::2], native_lon[::2],
           "terrain", "surface elevation [m]", vmin=-500, vmax=5500)
ax.plot(SITE_LON, SITE_LAT, "r*", ms=12, transform=ccrs.PlateCarree())
ax.set_title("ERA5 surface geopotential / $g_0$")
fig.subplots_adjust(left=0.04, right=0.96, top=0.92)

### Heights

MCEq integrates along a slant path through *geometric height*, so every level
needs one. Geopotential gives it directly:

$$ h = \frac{z}{g_0}, \qquad g_0 = 9.80665\ \mathrm{m/s^2} $$

which is why `geopotential` belongs in the CDS request next to `temperature`.
It is what makes the columns differ *geometrically* and not just thermally: in
this download the 1 hPa surface sits at 41.2 km over the South Pole and at
48.7 km over the Mediterranean.

Two properties of these heights are worth knowing before using them.

* They are **geopotential** heights, i.e. $z/g_0$. Geometric height differs by
  the variation of gravity with altitude, which is +0.3 % at 20 km and +0.8 %
  at 50 km. MCEq's own geometry is spherical, so this is well inside the
  accuracy of treating the atmosphere as horizontally layered.
* Below the terrain ERA5 **extrapolates**, and those levels are fictitious —
  over the Antarctic plateau the 1000 hPa surface comes out at $-55$ m, some
  2.9 km below the ice. Nothing in the pressure-level file marks where the
  ground is, so `TabulatedAtmosphere` cannot find it: left to itself it starts
  at the table's lowest non-negative height, i.e. essentially sea level. **For
  a site on high ground, pass `surface_elevation_m`** and the integration
  starts there instead.

**Without geopotential** the heights have to be rebuilt from the temperatures
with the hypsometric equation,

$$ \Delta z = \frac{R_d}{g_0}\,\bar{T}\,\ln\frac{p_\mathrm{low}}{p_\mathrm{up}}, $$

integrated upward from an anchor. The *thicknesses* are good — they use the
real temperatures — but the anchor is a guess: the fallback below puts the
1000 hPa surface at its US-Standard height of 111 m everywhere, which is a
rigid vertical shift of the whole column and flattens exactly the structure the
map further down is meant to show. The converter uses geopotential when present
and falls back to the integration only when it is not.

In [ ]:
R_D = 287.06  # gas constant of dry air, J/(kg K)
H_1000_HPA_M = 110.9  # US Standard height of the 1000 hPa surface


def column_heights(pressure_hpa, temperature_k, geopotential=None):
    """Geometric height of every pressure level, in metres.

    Args:
      pressure_hpa: (n_lev,) levels, ascending in height (descending pressure)
      temperature_k: (..., n_lev) temperatures
      geopotential: (..., n_lev) geopotential in m^2/s^2, or None

    Returns:
      (..., n_lev) heights above sea level in metres
    """
    if geopotential is not None:
        return geopotential / G0

    # Hypsometric integration upward from the bottom level.
    thickness = (
        R_D
        / G0
        * 0.5
        * (temperature_k[..., 1:] + temperature_k[..., :-1])
        * np.log(pressure_hpa[:-1] / pressure_hpa[1:])
    )
    base = np.full(temperature_k.shape[:-1] + (1,), H_1000_HPA_M)
    return np.concatenate([base, H_1000_HPA_M + np.cumsum(thickness, axis=-1)], axis=-1)


def write_gridded_table(filename, lat, lon, h_cm, t_k, p_hpa, comment):
    """Writes a v1 MCEq tabulated-atmosphere CSV holding a grid of columns.

    Args:
      lat: (n_lat,) latitudes, lon: (n_lon,) longitudes
      h_cm, t_k: (n_lat, n_lon, n_lev); p_hpa: (n_lev,)
    """
    n_lat, n_lon, n_lev = h_cm.shape
    lat_col = np.repeat(lat, n_lon * n_lev)
    lon_col = np.tile(np.repeat(lon, n_lev), n_lat)
    p_col = np.tile(p_hpa, n_lat * n_lon)
    rows = np.column_stack(
        [lat_col, lon_col, h_cm.reshape(-1), t_k.reshape(-1), p_col]
    )
    opener = gzip.open if filename.endswith(".gz") else open
    with opener(filename, "wt") as out:
        out.write("# MCEq tabulated atmosphere v1\n")
        out.write(f"# {comment}\n")
        out.write("lat_deg,lon_deg,h_cm,T_K,p_hPa\n")
        np.savetxt(out, rows, delimiter=",", fmt="%.4f,%.4f,%.6e,%.3f,%.6g")
    return filename

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

sub = ds.isel(latitude=slice(None, None, STRIDE), longitude=slice(None, None, STRIDE))
lat = sub["latitude"].values
lon = sub["longitude"].values

# Order the levels by ascending height, i.e. descending pressure.
p_hpa = np.sort(sub["pressure_level"].values)[::-1]

table_files = []
for step in range(sub.sizes["valid_time"]):
    day = sub.isel(valid_time=step)
    date = str(day["valid_time"].values)[:10]

    t_k = (
        day["t"]
        .sel(pressure_level=p_hpa)
        .transpose("latitude", "longitude", "pressure_level")
        .values.astype(np.float64)
    )
    z = (
        day["z"]
        .sel(pressure_level=p_hpa)
        .transpose("latitude", "longitude", "pressure_level")
        .values.astype(np.float64)
        if HAS_GEOPOTENTIAL
        else None
    )
    h_cm = column_heights(p_hpa, t_k, z) * 1e2

    path = write_gridded_table(
        os.path.join(OUTPUT_DIR, f"era5_{date}.csv.gz"),
        lat, lon, h_cm, t_k, p_hpa,
        f"ERA5 daily mean, {date}, {lat.size}x{lon.size} grid, "
        f"heights from {'geopotential' if HAS_GEOPOTENTIAL else 'hypsometric integration'}",
    )
    table_files.append(path)
    print(f"{path}  {os.path.getsize(path) / 1e6:5.1f} MB  "
          f"top level {h_cm[..., -1].mean() / 1e5:.1f} km (mean)")

### One site

`("Tabulated", (path, coord))` picks a single column out of the grid;
`("Tabulated_LC", (path, coord, depth_m))` binds the grid to a detector and
samples it at the shower impact point. Start with the plain column and compare
it to NRLMSISE-00 at the same place and season.

In [ ]:
import MCEq.geometry.density_profiles as dp

table = dp.AtmosphereTable.load_from_csv(table_files[0])
print(table)


def load_surface_table(filename):
    """Reads the surface table written above back into grids.

    Returns:
      (lat_axis, lon_axis, fields), fields a dict of (n_lat, n_lon) arrays
      keyed by column name
    """
    # Strip full-line comments first: numpy's names=True takes the *first*
    # line as the header whether it is a comment or not.
    with open(filename) as handle:
        rows = [line for line in handle if line.strip()
                and not line.lstrip().startswith("#")]
    data = np.genfromtxt(rows, delimiter=",", names=True, dtype=float)
    lat_axis, lon_axis = np.unique(data["lat_deg"]), np.unique(data["lon_deg"])
    jj = np.searchsorted(lat_axis, data["lat_deg"])
    ii = np.searchsorted(lon_axis, data["lon_deg"])
    fields = {}
    for name in data.dtype.names:
        if name in ("lat_deg", "lon_deg"):
            continue
        grid = np.full((lat_axis.size, lon_axis.size), np.nan)
        grid[jj, ii] = data[name]
        if np.isnan(grid).any():
            raise ValueError(f"{filename} does not cover a full grid")
        fields[name] = grid
    return lat_axis, lon_axis, fields


surface_lat, surface_lon, surface = load_surface_table(SURFACE_TABLE)


def surface_for(table, field="h_surface_m"):
    """A surface field re-indexed onto *table*'s own axes.

    Both grids hold the same cells, but nothing guarantees the same order --
    ERA5 counts latitude north-to-south, while load_from_csv sorts it
    south-to-north. Matching on the coordinates rather than trusting the
    positions is what keeps the Antarctic plateau out of the Arctic Ocean.
    """
    jj = [int(np.argmin(np.abs(surface_lat - value))) for value in table.lat_deg]
    ii = [int(np.argmin(np.abs(surface_lon - value))) for value in table.lon_deg]
    return surface[field][np.ix_(jj, ii)]


def cells_for(table):
    """The native orography points inside each cell of *table*'s grid.

    These are the elevations the column depth is averaged over. They come from
    the native field, so the atmosphere being on a coarser grid costs nothing
    here.
    """
    return (cell_lookup(table.lat_deg, native_lat),
            cell_lookup(table.lon_deg, native_lon, periodic=True))


table_elevation_m = surface_for(table)
table_cells = cells_for(table)
print(f"surface under the grid  {table_elevation_m.min():.1f} .. "
      f"{table_elevation_m.max():.1f} m (cell means)")
print(f"points per cell         "
      f"{len(table_cells[0][len(table_cells[0]) // 2]) * len(table_cells[1][0])}")

era5_atm = dp.TabulatedAtmosphere(
    table, coord=(SITE_LON, SITE_LAT), location=SITE_NAME, season="July"
)
msis_atm = dp.MSIS00LocationCentered(
    detector_coord=(SITE_LON, SITE_LAT), depth_m=SITE_DEPTH_M, season="July"
)
for atm in (era5_atm, msis_atm):
    atm.set_theta(0.0)
print(f"ERA5 vertical column {era5_atm.max_X:8.2f} g/cm^2")
print(f"MSIS vertical column {msis_atm.max_X:8.2f} g/cm^2")

#### Profiles

Geopotential gives every level a real height, so the natural vertical
coordinate is altitude. The sampling is still done in **slant depth** — `X` is
what MCEq integrates in, and `X2h` maps it back — so the curves are read out of
exactly the interface the solver uses: `X2rho`, `X2h`, `get_temperature`.

In [ ]:
X = np.geomspace(1.0, min(era5_atm.max_X, msis_atm.max_X) * 0.999, 300)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), dpi=120, sharey=True)


def profile(atm, X):
    """rho, T and height along a slant-depth grid, all via the interface."""
    h_cm = atm.X2h(X)
    temp = np.array([float(atm.get_temperature(h)) for h in h_cm])
    return atm.X2rho(X), temp, h_cm / 1e5


rho_e, t_e, h_e = profile(era5_atm, X)
rho_m, t_m, h_m = profile(msis_atm, X)

axes[0].plot(rho_e, h_e, "-", label="ERA5")
axes[0].plot(rho_m, h_m, "--", label="MSIS00")
axes[1].plot(t_e, h_e, "-", label="ERA5")
axes[1].plot(t_m, h_m, "--", label="MSIS00")

# Compared at equal depth, which is where the two models actually meet: the
# same X is the same overburden, whatever height each puts it at.
axes[2].plot(rho_e / rho_m, h_e, "-", label=r"$\rho$")
axes[2].plot(t_e / t_m, h_e, "--", label="T")
axes[2].axvline(1.0, color="0.6", lw=0.8)

# The geometric offset between the two is a difference, not a ratio: both
# heights go to zero at the ground, where a ratio says nothing.
off = axes[2].twiny()
off.plot((h_e - h_m) * 1e3, h_e, ":", color="tab:green", label="h")
off.set_xlabel("ERA5 - MSIS00 height at equal $X$ [m]", fontsize=9)
off.tick_params(labelsize=8)

axes[0].set_xscale("log")
axes[0].set_xlabel(r"$\rho$ [g/cm$^3$]")
axes[1].set_xlabel("T [K]")
axes[2].set_xlabel(r"ERA5 / MSIS00 at equal $X$ ($\rho$, T)")
axes[0].set_ylabel("height [km]")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle(f"{SITE_NAME}, {str(ds['valid_time'].values[0])[:10]}")
fig.tight_layout()

### Global maps, through the MCEq interface

The field below is read out of an MCEq atmosphere object built for each grid
node — not taken from the netCDF. That is the point: it exercises the same code
path a flux calculation uses, so what the map shows is what MCEq would
integrate.

**Vertical column depth** in g/cm², measured from the ground rather than from
sea level, so the map is dominated by orography: the Himalaya, the Andes, the
Antarctic plateau and Greenland stand out as deep minima, with the synoptic
pressure field as the finer structure over the ocean.

Each node's value is the column depth *averaged over its cell*, evaluated at
every native orography point inside it — the number the cell actually stands
for. That costs no extra atmospheres: `atm.h2X(h)` is the overburden above
height `h` for a column, and it reproduces rebuilding the same atmosphere with
`surface_elevation_m=h` to better than 10⁻³ g/cm², so one atmosphere per
node covers every elevation in its cell.

In [ ]:
def mceq_maps(table, cells=None, point_elev_m=None):
    """Vertical column depth above the ground at every grid node, via MCEq.

    One TabulatedAtmosphere per node, built at the *lowest* ground in its cell
    so its profile is never read below the terrain it was built for, then
    evaluated with h2X at every native orography point in the cell. The number
    comes from the same interface the solver integrates against.

    Args:
      table: gridded AtmosphereTable
      cells: (lat_indices, lon_indices) from cells_for(); recomputed when None
      point_elev_m: (n_lat, n_lon) single-point elevations for the comparison
        map; taken from the surface table when None

    Returns:
      (X_cell, X_point): two (n_lat, n_lon) arrays in g/cm^2 -- the average
      over the cell, and the value at the one native point nearest the node,
      which is what point-sampling the orography would have given
    """
    if cells is None:
        cells = cells_for(table)
    if point_elev_m is None:
        point_elev_m = surface_for(table, "h_point_m")
    jj_all, ii_all = cells
    n_lat, n_lon = table.lat_deg.size, table.lon_deg.size
    X_cell = np.empty((n_lat, n_lon))
    X_point = np.empty((n_lat, n_lon))
    for j, node_lat in enumerate(table.lat_deg):
        band = elevation_native_m[jj_all[j]]
        for i, node_lon in enumerate(table.lon_deg):
            # MCEq's geometry measures altitude from sea level and will not
            # take a negative observation level, so the basins below it (Dead
            # Sea, Caspian, Turfan) are clamped and come out too shallow by the
            # air they are missing -- some 43 g/cm^2 at the -360 m of the Dead
            # Sea, which the native orography does resolve.
            cell = np.maximum(band[:, ii_all[i]].ravel(), 0.0)
            atm = dp.TabulatedAtmosphere(
                table, coord=(node_lon, node_lat),
                surface_elevation_m=float(cell.min()),
            )
            atm.set_theta(0.0)
            X_cell[j, i] = float(np.mean(atm.h2X(cell * 1e2)))
            X_point[j, i] = float(atm.h2X(max(point_elev_m[j, i], 0.0) * 1e2))
    return X_cell, X_point


import time

start = time.time()
max_X, max_X_point = mceq_maps(table, table_cells)
print("%d columns through MCEq in %.1f s" % (max_X.size, time.time() - start))
print("column depth     %.1f .. %.1f g/cm^2" % (max_X.min(), max_X.max()))

In [ ]:
fig = plt.figure(figsize=(9, 5.0), dpi=120)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())
global_map(ax, max_X, table.lat_deg, table.lon_deg, "viridis",
           "vertical column depth max_X [g/cm$^2$]")
ax.plot(SITE_LON, SITE_LAT, "r*", ms=12, transform=ccrs.PlateCarree())
ax.set_title(f"ERA5 through MCEq — {str(ds['valid_time'].values[0])[:10]}")
fig.subplots_adjust(left=0.04, right=0.96, top=0.92)

#### What averaging the cell is worth

The same map with the ground point-sampled instead — one native orography node
per cell, the obvious way to build it — differs by the amounts below. This is
the resolution that matters here: an order of magnitude more than a finer
*atmospheric* grid would buy, and it is free, because the elevations are
aggregated from the native field at whatever `STRIDE` is set to.

The median land cell barely moves — most land is flat — but the 95th
percentile is ~27 g/cm² and the worst cells are off by ~167. And it is a bias,
not noise: a node that happens to sit on a summit is too shallow for its whole
cell and one in a valley too deep, so the error does not average out over a
region, it follows the terrain. Over the ocean, where there is no terrain, the
two agree to 0.1 g/cm².

In [ ]:
diff = max_X_point - max_X
land = table_elevation_m > 1.0
print("point-sampled ground minus cell average, over land (%d cells):" % land.sum())
print("  median %+6.1f g/cm^2    p95 |diff| %5.1f    max |diff| %5.1f"
      % (np.median(diff[land]), np.percentile(np.abs(diff[land]), 95),
         np.abs(diff[land]).max()))
print("over ocean:            p95 |diff| %5.2f g/cm^2"
      % np.percentile(np.abs(diff[~land]), 95))

fig = plt.figure(figsize=(9, 5.0), dpi=120)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())
lim = np.percentile(np.abs(diff), 99.5)
global_map(ax, diff, table.lat_deg, table.lon_deg, "RdBu_r",
           r"point-sampled $-$ cell-averaged ground [g/cm$^2$]",
           vmin=-lim, vmax=lim)
ax.set_title("what point-sampling the orography would cost")
fig.subplots_adjust(left=0.04, right=0.96, top=0.92)

#### A site on high ground

The map above starts every column at sea level. For a detector on the Antarctic
plateau that is 2.8 km of atmosphere that is not there — `surface_elevation_m`
is what removes it, and the difference is large enough to matter for any rate
calculation.

In [ ]:
SOUTH_POLE = (0.0, -90.0)  # lon, lat
ICECUBE_ELEVATION_M = 2835.0

sea_level = dp.TabulatedAtmosphere(table, coord=SOUTH_POLE, location="SouthPole")
on_the_ice = dp.TabulatedAtmosphere(
    table, coord=SOUTH_POLE, surface_elevation_m=ICECUBE_ELEVATION_M,
    location="SouthPole",
)
msis_ic = dp.MSIS00Atmosphere("SouthPole", "July")
for atm in (sea_level, on_the_ice, msis_ic):
    atm.set_theta(0.0)

print(f"ERA5, default (sea level)      {sea_level.max_X:7.2f} g/cm^2  "
      f"h_obs {sea_level.geom.h_obs / 1e5:.3f} km")
print(f"ERA5, surface_elevation_m set  {on_the_ice.max_X:7.2f} g/cm^2  "
      f"h_obs {on_the_ice.geom.h_obs / 1e5:.3f} km")
print(f"MSIS00 SouthPole               {msis_ic.max_X:7.2f} g/cm^2  "
      f"h_obs {msis_ic.geom.h_obs / 1e5:.3f} km")

#### Day to day

The same map for every day in the download, as the **anomaly** from the mean
over those days. The absolute field is not the useful thing to animate: it spans
about 100 g/cm² across the globe while a typical grid node moves less than
1 g/cm² from one day to the next, so on a common colour scale the animation
looks frozen. The departure from the mean is where the weather is — the
mid-latitude storm tracks swing by tens of g/cm², and that day-to-day motion of
the overburden is exactly what a seasonal-variation analysis has to fold in.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

days = []
for path in table_files:
    day_table = dp.AtmosphereTable.load_from_csv(path)
    days.append(mceq_maps(day_table, table_cells)[0])
    print("done", path)

stack = np.array(days)
anomaly = stack - stack.mean(axis=0)
dates = [os.path.basename(p).replace("era5_", "").split(".")[0]
         for p in table_files]

# Symmetric about zero so the colour scale reads as a departure, not a value.
lim = np.percentile(np.abs(anomaly), 99)
print("column depth   %.1f .. %.1f g/cm^2 across the globe" % (stack.min(), stack.max()))
print("day-to-day     %.2f g/cm^2 for the median node, %.1f at the 99th pct"
      % (np.median(stack.max(0) - stack.min(0)), np.percentile(anomaly, 99)))

fig = plt.figure(figsize=(8, 4.5), dpi=110)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())
mesh = global_map(ax, anomaly[0], table.lat_deg, table.lon_deg, "RdBu_r",
                  rf"max_X $-$ {len(dates)}-day mean [g/cm$^2$]",
                  vmin=-lim, vmax=lim)
ax.plot(SITE_LON, SITE_LAT, "k*", ms=11, transform=ccrs.PlateCarree())
title = ax.set_title("")


def draw(step):
    cyclic, _ = add_cyclic_point(anomaly[step], coord=table.lon_deg)
    mesh.set_array(cyclic.ravel())
    title.set_text(dates[step])
    return mesh, title


ani = animation.FuncAnimation(fig, draw, frames=len(dates), interval=700, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())

### Why the grid matters for neutrinos

`TabulatedLocationCentered` follows the shower axis from the detector toward
the source and takes the table column where it crosses the surface. For a
downgoing shower that is near the detector. For an **upgoing** one — the signal
region of a neutrino telescope — the axis passes through the Earth and the
shower developed on the *far side*, in an atmosphere that has nothing to do
with the one overhead. A single-column table cannot express that; a global grid
can, which is why `max_theta=180` needs one.

The map below traces the impact point over the whole sky, coloured by the slant
depth MCEq computes there.

> **Open question — the elevation at the impact point.**
> The atmospheric *column* follows the impact point, but the *ground* under it
> does not. `TabulatedLocationCentered` uses one observation level for the
> whole sky: the detector's own elevation for downgoing showers, and sea level
> for the upgoing ones that develop on the far side. So a shower whose impact
> point lands on the Antarctic plateau or the Tibetan plateau is still
> integrated from the detector's elevation, and it picks up the couple of
> hundred g/cm² of air that the terrain there actually displaces.
>
> The surface table built at the top of this notebook is exactly what would fix
> it — one elevation per node, ready to be looked up at the impact point — but
> wiring it in raises questions that should be settled first rather than
> guessed at. Which elevation is even right for a shower that crosses a
> mountain range at 20° above the horizon, when the column is not vertical and
> the ground under it varies by kilometres along the path? At what zenith angle
> does the flat-Earth-under-the-impact-point picture stop being the dominant
> error compared with the curvature the geometry already handles? And should
> the surface elevation live in the atmosphere table itself, as another column
> per node, rather than in a table of its own?
>
> Until that is worked out, treat overburden for impact points over high
> terrain as approximate. It does not affect the vertical maps above, which set
> each node's elevation explicitly.

In [ ]:
lc_atm = dp.TabulatedLocationCentered(
    table,
    detector_coord=(SITE_LON, SITE_LAT),
    depth_m=SITE_DEPTH_M,
    max_theta=180.0,
    location=SITE_NAME,
    season="July",
)

zeniths = np.arange(0.0, 180.1, 7.5)
azimuths = np.arange(0.0, 360.0, 45.0)
track = []
for azimuth in azimuths:
    for zenith in zeniths:
        lc_atm.set_theta(float(zenith), azimuth_deg=float(azimuth))
        track.append(
            (lc_atm.current_impact_longitude, lc_atm.current_impact_latitude,
             lc_atm.max_X, zenith, azimuth)
        )
track = np.array(track)
print(f"{len(track)} directions, slant depth "
      f"{track[:, 2].min():.0f} .. {track[:, 2].max():.0f} g/cm^2")

In [ ]:
fig = plt.figure(figsize=(13, 5.2), dpi=120)
downgoing = track[:, 3] <= 90.0

# --- the whole sky ---------------------------------------------------------
ax = fig.add_subplot(1, 2, 1, projection=ccrs.Robinson())
global_map(ax, max_X, table.lat_deg, table.lon_deg, paled("viridis"),
           "vertical column depth max_X [g/cm$^2$]")
for mask, marker, name in (
    (downgoing, "o", "downgoing"),
    (~downgoing, "^", "upgoing (far side)"),
):
    scat = ax.scatter(
        track[mask, 0], track[mask, 1], c=np.log10(track[mask, 2]),
        s=22, marker=marker, cmap="viridis", vmin=np.log10(track[:, 2]).min(),
        vmax=np.log10(track[:, 2]).max(), transform=ccrs.PlateCarree(),
        zorder=3, edgecolors="k", linewidths=0.25, label=name,
    )
ax.plot(SITE_LON, SITE_LAT, "r*", ms=15, transform=ccrs.PlateCarree(), zorder=4)
plt.colorbar(scat, ax=ax, orientation="vertical", pad=0.02,
             label=r"$\log_{10}$ slant depth [g/cm$^2$]")
ax.legend(loc="lower left", fontsize=8)
ax.set_title(f"All sky from {SITE_NAME} (star):\nupgoing showers develop on the far side", fontsize=10)

# --- and the downgoing cluster, which the star hides above ------------------
ax2 = fig.add_subplot(1, 2, 2, projection=ccrs.PlateCarree())
near = ax2.scatter(
    track[downgoing, 0], track[downgoing, 1], c=track[downgoing, 3],
    s=26, cmap="viridis", transform=ccrs.PlateCarree(), zorder=3,
    edgecolors="k", linewidths=0.25,
)
ax2.plot(SITE_LON, SITE_LAT, "r*", ms=15, transform=ccrs.PlateCarree(), zorder=4)
ax2.set_extent([SITE_LON - 3.5, SITE_LON + 3.5, SITE_LAT - 3.0, SITE_LAT + 3.0],
               crs=ccrs.PlateCarree())
ax2.coastlines(lw=0.5, color="0.3")
grid = ax2.gridlines(draw_labels=True, lw=0.3, color="0.7")
grid.top_labels = grid.right_labels = False
plt.colorbar(near, ax=ax2, orientation="vertical", pad=0.02,
             label="zenith angle [deg]")
ax2.set_title("Downgoing: the column drifts\nwith zenith and azimuth", fontsize=10)

# tight_layout does not cope with cartopy gridline labels
fig.subplots_adjust(left=0.02, right=0.97, top=0.86, wspace=0.28)

### Where to go from here

* **Humidity.** Everything above is dry air. `specific_humidity` is on the same
  pressure levels; folding it in changes the density by less than a percent
  near the surface and less above, so it matters only for precision work.
* **Resolution.** Two different fields hide under that word and they do not
  cost the same. The **ground** is aggregated from the native orography at any
  `STRIDE`, which is the part that matters: point-sampling it instead costs
  27 g/cm² at the 95th percentile over land and up to 167 g/cm². The
  **atmosphere** is strided, and going back from 1 deg to the native 0.25 deg
  buys 0.3 g/cm² over the Mediterranean and 1.7 g/cm² over Tibet (p95) against
  a ~8 g/cm² day-to-day spread — less than MCEq's own one-dimensional
  approximation, which develops the shower in a single column while the shower
  maximum sits ~126 km downrange at 80 deg zenith. For a regional analysis crop
  first and then keep every point: a 20x20 deg box at 0.25 deg is 11 MB and
  loads in 0.3 s. A *global* native table is not reachable through CSV at all —
  38 M rows, 1.7 GB, and ~30 GB of RAM in the loader — and fixing that means a
  binary format rather than a finer stride.
* **Seasonal studies.** One table per day; build one atmosphere per table and
  pass them to `MCEqRun.solve_batch(..., conditions=...)` as per-member
  `density_model`s to get a time series in one call.
* **Other datasets.** Only the converter above is ERA5-specific. Write
  `h_cm` plus `T_K`/`p_hPa` (or `rho_gcm3`), optionally `lat_deg`/`lon_deg`,
  and any archive works.